# Enforcing a Valid Classification Vocabulary (with Retry)

When you classify pages with a fixed set of classes (e.g. `invoice`, `w2`,
`check`), a language model can occasionally return a label that is **not** in
your configured list — for example predicting `receipt` when only `invoice`,
`w2`, and `check` are valid. Smaller / cheaper models are especially prone to
this.

The `multimodalPageLevelClassification` method in `idp_common` supports a
**deterministic validation + retry loop** that fixes this:

1. After the model returns a class, validate it against the configured vocabulary.
2. If it is **not** valid, re-prompt the model — appending a correction message
   that lists the allowed classes (this matters: at `temperature=0`, re-sending
   the *same* request would return the *same* invalid answer).
3. Retry up to `maxValidationRetries` times.
4. If retries are exhausted, assign the configured `invalidClassFallback`
   (default `unclassified`) and flag the page with a `validation_error`.

This notebook demonstrates the behavior **deterministically** by mocking the
Bedrock call so we can force an out-of-vocabulary prediction. In production the
same loop runs against the real model — no code changes required, just the
config flags shown below.

> **Related config keys** (under `classification:`):
> - `enforceValidClasses` (default `true`)
> - `maxValidationRetries` (default `2`)
> - `invalidClassFallback` (default `unclassified`)


## 1. Imports

In [1]:
import json
import logging
from unittest.mock import patch

from idp_common.classification.service import ClassificationService

# Surface the retry/validation log messages so we can see the loop work.
logging.basicConfig(level=logging.WARNING)
logging.getLogger("idp_common.classification").setLevel(logging.INFO)

print("Imports OK")

Imports OK


## 2. Define a configuration with a fixed class vocabulary

We define three valid classes and enable enforcement with up to 2 retries.


In [2]:
def make_class(name, description):
    return {
        "$schema": "https://json-schema.org/draft/2020-12/schema",
        "$id": name,
        "x-aws-idp-document-type": name,
        "type": "object",
        "description": description,
        "properties": {},
    }

config = {
    "classes": [
        make_class("invoice", "An invoice document"),
        make_class("w2", "A W-2 tax form"),
        make_class("check", "A bank check"),
    ],
    "classification": {
        "model": "us.amazon.nova-lite-v1:0",
        "temperature": 0.0,
        "top_k": 5,
        "system_prompt": "You are a document classification assistant.",
        "task_prompt": (
            "Classify the document into one of:\n"
            "{CLASS_NAMES_AND_DESCRIPTIONS}\n\n"
            "Document text:\n{DOCUMENT_TEXT}\n\n"
            "Image:\n{DOCUMENT_IMAGE}\n\n"
            'Respond with JSON: {"class": "<type>"}'
        ),
        "classificationMethod": "multimodalPageLevelClassification",
        # --- the feature under test ---
        "enforceValidClasses": True,
        "maxValidationRetries": 2,
        "invalidClassFallback": "unclassified",
    },
}

print("Valid classes:", [c["$id"] for c in config["classes"]])

Valid classes: ['invoice', 'w2', 'check']


## 3. A helper to simulate the model

Real models won't reliably emit an invalid class on demand, so we mock
`_invoke_bedrock_model` to return a scripted sequence of responses. This lets
us demonstrate each scenario deterministically. Each mocked response uses the
same shape the real Bedrock client returns.


In [3]:
def bedrock_response(class_value):
    """Build a fake Bedrock Converse response for the given class label."""
    return {
        "response": {
            "output": {
                "message": {"content": [{"text": json.dumps({"class": class_value})}]}
            }
        },
        "metering": {"bedrock": {"inputTokens": 120, "outputTokens": 8}},
    }


def run_with_responses(cfg, responses):
    """Classify one page, feeding the scripted `responses` to the model."""
    with patch("boto3.Session"):
        service = ClassificationService(region="us-west-2", config=cfg, backend="bedrock")

    with (
        patch("idp_common.s3.get_text_content", return_value="ACME Corp Invoice #42  Total: $100"),
        patch("idp_common.image.prepare_image", return_value=b"img"),
        patch("idp_common.image.prepare_bedrock_image_attachment", return_value={"image": "b64"}),
        patch.object(ClassificationService, "_invoke_bedrock_model", side_effect=responses) as mock_invoke,
    ):
        result = service.classify_page_bedrock(
            page_id="1",
            text_uri="s3://bucket/text.txt",
            image_uri="s3://bucket/image.jpg",
        )
    return result, mock_invoke.call_count

## 4. Scenario A — model corrects itself on retry

The model first returns `receipt` (not in our vocabulary). The validation loop
re-prompts, and on the second attempt the model returns the valid `invoice`.


In [4]:
result, calls = run_with_responses(
    config,
    [bedrock_response("receipt"), bedrock_response("invoice")],
)

print(f"\nFinal class : {result.classification.doc_type}")
print(f"Model calls : {calls}  (1 initial + 1 retry)")
print(f"validation_error present: {'validation_error' in result.classification.metadata}")
assert result.classification.doc_type == "invoice"
assert calls == 2

INFO:idp_common.classification.service:Classification caching disabled
INFO:idp_common.classification.service:Initialized classification service with Bedrock backend using model us.amazon.nova-lite-v1:0
INFO:idp_common.classification.service:Using multimodal page-level classification method with document boundary detection
INFO:idp_common.classification.service:Classifying page 1 with Bedrock
INFO:idp_common.classification.service:Parsed classification response as json: {'class': 'receipt'}
INFO:idp_common.classification.service:Parsed classification response as json: {'class': 'invoice'}
INFO:idp_common.classification.service:Time taken for classification of page 1: 0.00 seconds
INFO:idp_common.classification.service:Page 1 classified as invoice



Final class : invoice
Model calls : 2  (1 initial + 1 retry)
validation_error present: False


## 5. Scenario B — retries exhausted, fallback applied

The model returns an invalid class on every attempt. After the initial call
plus 2 retries (3 total), the page is assigned the `invalidClassFallback`
(`unclassified`) and flagged with a `validation_error`.


In [5]:
result, calls = run_with_responses(
    config,
    [bedrock_response("receipt")] * 5,  # always invalid; only 3 will be consumed
)

print(f"\nFinal class : {result.classification.doc_type}")
print(f"Model calls : {calls}  (1 initial + 2 retries)")
print(f"validation_error: {result.classification.metadata.get('validation_error')}")
assert result.classification.doc_type == "unclassified"
assert calls == 3

INFO:idp_common.classification.service:Classification caching disabled
INFO:idp_common.classification.service:Initialized classification service with Bedrock backend using model us.amazon.nova-lite-v1:0
INFO:idp_common.classification.service:Using multimodal page-level classification method with document boundary detection
INFO:idp_common.classification.service:Classifying page 1 with Bedrock
INFO:idp_common.classification.service:Parsed classification response as json: {'class': 'receipt'}
INFO:idp_common.classification.service:Parsed classification response as json: {'class': 'receipt'}
INFO:idp_common.classification.service:Parsed classification response as json: {'class': 'receipt'}
ERROR:idp_common.classification.service:Page 1: Model returned invalid class 'receipt' after 3 attempt(s); assigned fallback 'unclassified'.
INFO:idp_common.classification.service:Time taken for classification of page 1: 0.00 seconds
INFO:idp_common.classification.service:Page 1 classified as unclassifi


Final class : unclassified
Model calls : 3  (1 initial + 2 retries)
validation_error: Model returned invalid class 'receipt' after 3 attempt(s); assigned fallback 'unclassified'.


## 6. Scenario C — enforcement disabled (legacy behavior)

With `enforceValidClasses: false`, an out-of-vocabulary prediction is logged as
a warning and used **as-is** — no retry, no fallback. This is the behavior prior
to this feature, retained for backward compatibility.


In [6]:
legacy_config = json.loads(json.dumps(config))  # deep copy
legacy_config["classification"]["enforceValidClasses"] = False

result, calls = run_with_responses(legacy_config, [bedrock_response("receipt")])

print(f"\nFinal class : {result.classification.doc_type}  (used as-is)")
print(f"Model calls : {calls}")
assert result.classification.doc_type == "receipt"
assert calls == 1

INFO:idp_common.classification.service:Classification caching disabled
INFO:idp_common.classification.service:Initialized classification service with Bedrock backend using model us.amazon.nova-lite-v1:0
INFO:idp_common.classification.service:Using multimodal page-level classification method with document boundary detection
INFO:idp_common.classification.service:Classifying page 1 with Bedrock
INFO:idp_common.classification.service:Parsed classification response as json: {'class': 'receipt'}
INFO:idp_common.classification.service:Time taken for classification of page 1: 0.00 seconds
INFO:idp_common.classification.service:Page 1 classified as receipt



Final class : receipt  (used as-is)
Model calls : 1


## 7. Summary

| Scenario | `enforceValidClasses` | Model behavior | Result |
|----------|----------------------|----------------|--------|
| A | `true` | invalid → valid on retry | corrected to valid class |
| B | `true` | always invalid | `invalidClassFallback` + `validation_error` |
| C | `false` | invalid | invalid class used as-is (legacy) |

**To enable in your deployment**, set these under `classification:` in your
config (they are on by default for new deployments):

```yaml
classification:
  enforceValidClasses: true
  maxValidationRetries: 2
  invalidClassFallback: unclassified
```

These are also editable in the **Configuration UI** under the Classification
section.

> **Note:** This applies to `multimodalPageLevelClassification`. Holistic
> packet classification has similar needs but is not covered by this loop yet.
